# 02 — Train Global Topic Model
Incrementally train a BERTopic model across 5 shuffled batches, classify all documents, and store global topic assignments in DuckDB.

In [ ]:
import sys
sys.path.insert(0, '../src')

import yaml
from pathlib import Path

from reddit_np_topics.db import get_connection, read_table, write_dataframe, table_row_count
from reddit_np_topics.modeling.train_global import train_incremental, classify_with_base_model, save_model
from reddit_np_topics.modeling.utils import get_merge_candidates, attach_topic_info

with open('../config.yaml') as f:
    cfg = yaml.safe_load(f)

conn = get_connection(Path('..') / cfg['paths']['database'])

In [ ]:
# Load normalized data and shuffle
df = read_table(conn, 'normalized_posts')
df = df.sample(frac=1, random_state=42).reset_index(drop=True)
print(f'Loaded {len(df)} documents across {df.park_name.nunique()} parks')

In [ ]:
# Step 1: Incremental training across N batches
base_model = train_incremental(
    docs=df['tokens'],
    cfg=cfg['modeling'],
    n_batches=cfg['modeling']['global']['n_batches'],
)
base_model.get_topic_info()

In [ ]:
# Inspect similar topics — candidates for merging
get_merge_candidates(base_model, threshold=0.85)

In [ ]:
# Step 2: Manually define topic merges based on inspection above
# Edit this list based on your model's output
topics_to_merge = [
    # [1, 88],   # example: Access/Roads
    # [25, 31, 39, 7, 67],  # example: Camping
]

if topics_to_merge:
    base_model.merge_topics(df['tokens'].tolist(), topics_to_merge)

base_model.get_topic_info()

In [ ]:
# Step 3: Classify full dataset using base model topic assignments
final_model, topics, probs = classify_with_base_model(
    docs=df['tokens'],
    base_model=base_model,
    cfg=cfg['modeling'],
)
final_model.get_topic_info()

In [ ]:
# Step 4: Set human-readable topic labels
# Edit these labels based on your model's topics
topic_labels = {
    0: "Hiking Recommendations",
    1: "Camping",
    2: "Photography",
    3: "Accessibility",
    # ... add all topics
}
final_model.set_topic_labels(topic_labels)
final_model.get_topic_info()

In [ ]:
# Step 5: Attach topic info to documents and save to DuckDB
result_df = attach_topic_info(
    df, final_model, topics,
    id_col='Global_Topic_ID',
    name_col='Global_Topic',
    repr_col='Global_Repr',
)

write_dataframe(conn, result_df, 'global_classified', mode='replace')
print(f'global_classified: {table_row_count(conn, "global_classified")} rows')

In [ ]:
# Save the model
save_model(final_model, Path('..') / cfg['paths']['models_global'], 'global_model')